This takes the Detached home dataset and identifies the ones that have hot 
water flow temperature measurements for the final training dataset.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [3]:
df_detached_homes = pd.read_csv(
    "../retrieved_weather_data/consumption_weather_merged.csv")

In [6]:
id_names = df_detached_homes["Property_ID"].unique()

In [49]:
id_missing = []
for id in id_names:
    df = df_detached_homes[df_detached_homes["Property_ID"]==id]
    missing_counts = df_detached_homes[df_detached_homes["Property_ID"]==
    id][["Hot_Water_Flow_Temperature"]].isna().sum()
    ratio = int(missing_counts.iloc[0])/len(df)
    if ratio == 1.0: 
        id_missing.append(id)

print(id_missing)

['EOH0018', 'EOH0245', 'EOH0258', 'EOH0413', 'EOH0590', 'EOH0749', 'EOH0930', 'EOH1003', 'EOH1046', 'EOH1063', 'EOH1416', 'EOH1512', 'EOH1566', 'EOH1619', 'EOH1658', 'EOH1751', 'EOH1752', 'EOH1792', 'EOH1794', 'EOH1884', 'EOH1908', 'EOH1980', 'EOH2045', 'EOH2089', 'EOH2198', 'EOH2205', 'EOH2264', 'EOH2308', 'EOH2310', 'EOH2512', 'EOH2546', 'EOH2619', 'EOH2790', 'EOH2852', 'EOH2919', 'EOH2951', 'EOH2963', 'EOH3062', 'EOH3148']


In [63]:
df_detached_homes_final = df_detached_homes[~df_detached_homes["Property_ID"]
                          .isin(id_missing)]

In [67]:
df_detached_homes_final

,Unnamed: 0,Property_ID,Timestamp,half-hour,Boiler_Energy_Output,Circulation_Pump_Energy_Consumed,Heat_Pump_Energy_Output,Whole_System_Energy_Consumed,External_Air_Temperature,Heat_Pump_Heating_Flow_Temperature,...,Brine_Return_Temperature,Postcode,Postcode_Timestamp,temp,humidity,windspeed,solarradiation,pressure,visibility,cloudcover
0,0,EOH0005,2021-05-21 12:00:00,12:00:00,NaN,0.003,NaN,0.240,8.67,NaN,...,NaN,EH33,EH33_2021-05-21 12:00:00,9.0,94.430,26.30,84.0,990.40,5.90,96.00
1,1,EOH0005,2021-05-21 12:30:00,12:30:00,NaN,0.020,NaN,1.782,8.58,NaN,...,NaN,EH33,EH33_2021-05-21 12:30:00,9.0,94.180,26.90,77.0,990.70,6.00,96.05
2,2,EOH0005,2021-05-21 13:00:00,13:00:00,NaN,0.036,NaN,1.852,8.50,NaN,...,NaN,EH33,EH33_2021-05-21 13:00:00,9.0,93.930,27.50,70.0,991.00,6.10,96.10
3,3,EOH0005,2021-05-21 13:30:00,13:30:00,NaN,0.038,NaN,2.466,8.50,NaN,...,NaN,EH33,EH33_2021-05-21 13:30:00,8.9,93.535,26.40,67.5,991.30,6.70,92.90
4,4,EOH0005,2021-05-21 14:00:00,14:00:00,NaN,0.038,2.514,3.323,8.52,NaN,...,NaN,EH33,EH33_2021-05-21 14:00:00,8.8,93.140,25.30,65.0,991.60,7.30,89.70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11113027,11113027,EOH3204,2023-03-10 09:00:00,09:00:00,NaN,NaN,NaN,NaN,2.96,NaN,...,NaN,RH18,RH18_2023-03-10 09:00:00,3.5,92.570,24.30,67.0,991.90,5.90,100.00
11113028,11113028,EOH3204,2023-03-10 09:30:00,09:30:00,NaN,NaN,NaN,NaN,2.73,NaN,...,NaN,RH18,RH18_2023-03-10 09:30:00,3.2,91.205,25.05,89.5,993.55,11.45,99.70
11113029,11113029,EOH3204,2023-03-10 10:00:00,10:00:00,NaN,NaN,NaN,NaN,2.83,NaN,...,NaN,RH18,RH18_2023-03-10 10:00:00,2.9,89.840,25.80,112.0,995.20,17.00,99.40
11113030,11113030,EOH3204,2023-03-10 10:30:00,10:30:00,NaN,NaN,NaN,NaN,3.31,NaN,...,NaN,RH18,RH18_2023-03-10 10:30:00,3.2,86.745,26.90,163.0,996.55,31.30,99.70


In [70]:
df_detached_homes_final["Timestamp"] = pd.to_datetime(
    df_detached_homes_final["Timestamp"]
)

/var/folders/fk/0m0g9qgs7y5fc0br86wht_1m0000gn/T/ipykernel_33386/1124800155.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_detached_homes_final["Timestamp"] = pd.to_datetime(


In [71]:
range_df = (
    df_detached_homes_final
    .dropna(subset=["Hot_Water_Flow_Temperature"])
    .groupby("Property_ID")["Timestamp"]
    .agg(
        first_available="min",
        last_available="max"
    )
)

range_df["range_days"] = (
    range_df["last_available"] - range_df["first_available"]
).dt.days

In [73]:
counts = (
    df_detached_homes_final
    .groupby("Property_ID")["Hot_Water_Flow_Temperature"]
    .count()
    .rename("num_samples")
)

range_df = range_df.join(counts)


In [74]:
range_df.reset_index()


,Property_ID,first_available,last_available,range_days,num_samples
0,EOH0005,2021-05-21 14:00:00,2023-09-28 15:00:00,860,3715
1,EOH0021,2021-08-17 15:30:00,2023-09-27 19:30:00,771,1395
2,EOH0026,2021-05-06 08:30:00,2023-09-28 19:30:00,875,3043
3,EOH0033,2021-07-07 11:00:00,2023-09-28 06:30:00,812,2279
4,EOH0043,2021-06-28 13:30:00,2022-06-14 13:30:00,351,980
...,...,...,...,...,...
256,EOH3167,2021-06-08 17:30:00,2023-09-28 17:00:00,841,4272
257,EOH3186,2021-06-30 14:00:00,2023-09-29 00:00:00,820,25086
258,EOH3196,2020-11-27 14:00:00,2023-09-28 14:30:00,1035,6638
259,EOH3197,2021-07-27 18:30:00,2022-03-31 08:30:00,246,881


In [ ]:
id_low_data =[]
for id in range_df.index.unique():
    df = range_df[range_df.index==id]
    if df["range_days"].iloc[0] <= 100:
        id_low_data.append(id)

print(id_low_data)

In [ ]:
df_detached_homes_v2 = df_detached_homes_final[~df_detached_homes_final
["Property_ID"].isin(id_low_data)]

In [87]:
range_df_hp = (
    df_detached_homes_v2
    .dropna(subset=["Heat_Pump_Energy_Output"])
    .groupby("Property_ID")["Timestamp"]
    .agg(
        first_available="min",
        last_available="max"
    )
)

range_df_hp["range_days"] = (
    range_df_hp["last_available"] - range_df_hp["first_available"]
).dt.days


range_df_hw = (
    df_detached_homes_v2
    .dropna(subset=["Hot_Water_Flow_Temperature"])
    .groupby("Property_ID")["Timestamp"]
    .agg(
        first_available="min",
        last_available="max"
    )
)

range_df_hw["range_days_hw"] = (
    range_df_hw["last_available"] - range_df_hw["first_available"]
).dt.days

counts_hp = (
    df_detached_homes_v2
    .groupby("Property_ID")["Heat_Pump_Energy_Output"]
    .count()
    .rename("num_samples_hp")
)

counts_hw = (
    df_detached_homes_v2
    .groupby("Property_ID")["Hot_Water_Flow_Temperature"]
    .count()
    .rename("num_samples_hot_water")
)

range_df_hp = range_df_hp.join(range_df_hw["range_days_hw"])
range_df_hp = range_df_hp.join(counts_hp)
range_df_hp =range_df_hp.join(counts_hw)
range_df_hp.reset_index()

,Property_ID,first_available,last_available,range_days,range_days_hw,num_samples_hp,num_samples_hot_water
0,EOH0005,2021-05-21 14:00:00,2023-09-28 23:30:00,860,860,33016,3715
1,EOH0021,2021-08-17 15:30:00,2023-09-28 23:30:00,772,771,34826,1395
2,EOH0026,2021-05-06 00:00:00,2023-09-28 23:30:00,875,875,41932,3043
3,EOH0033,2021-07-07 10:30:00,2023-09-28 23:30:00,813,812,32522,2279
4,EOH0043,2021-06-28 12:30:00,2022-06-14 23:30:00,351,351,16785,980
...,...,...,...,...,...,...,...
252,EOH3167,2021-02-25 15:30:00,2023-09-28 23:30:00,945,841,45260,4272
253,EOH3186,2021-06-30 12:30:00,2023-09-28 23:30:00,820,820,37183,25086
254,EOH3196,2021-05-19 00:00:00,2023-09-28 23:30:00,862,1035,41321,6638
255,EOH3197,2021-07-27 18:00:00,2022-03-31 23:30:00,247,246,11833,881


In [92]:
id_low_data_hp =[]
for id in range_df_hp.index.unique():
    df = range_df_hp[range_df_hp.index==id]
    if df["range_days"].iloc[0] <= 365*1.25:
        id_low_data_hp.append(id)

print(id_low_data_hp)

['EOH0043', 'EOH0132', 'EOH0205', 'EOH0241', 'EOH0465', 'EOH0728', 'EOH0735', 'EOH0830', 'EOH1053', 'EOH1135', 'EOH1190', 'EOH1285', 'EOH1287', 'EOH1400', 'EOH1659', 'EOH1740', 'EOH1870', 'EOH1873', 'EOH2216', 'EOH2390', 'EOH2421', 'EOH2505', 'EOH2540', 'EOH2700', 'EOH2713', 'EOH2750', 'EOH2751', 'EOH2758', 'EOH2956', 'EOH3021', 'EOH3197']


In [93]:
df_detached_homes_v3 = df_detached_homes_v2[~df_detached_homes_v2
["Property_ID"].isin(id_low_data_hp)]

In [96]:
(df_detached_homes_v3.to_parquet
 ("retrieved_weather_data/merged_data_final_homes.parquet"))